# 11.12 像素寫的0與標籤字串0，怎麼對應？


下面兩張五行、三欄的字形，左邊寫數字 0，右邊寫數字 1。希望讀字結果是字串 `"0"`、`"1"`。先把這些真正可見的筆畫與標籤配好，再談辨識器。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-11-glyph-labels.svg")))

In [ ]:
import torch

glyphs = {"0": ["111", "101", "101", "101", "111"], "1": ["010", "110", "010", "010", "111"]}
for label, rows in glyphs.items():
    image = torch.tensor([[int(c) for c in row] for row in rows], dtype=torch.float32)
    print("文字標籤", label, "像素形狀", tuple(image.shape))
    print(image)

字典的鍵是正確文字標籤，值是五行字形。行內 `"1"` 轉成像素亮度 1、`"0"` 轉成背景 0；每張 tensor `(5,3)`。因此「像素 0」表示暗，而「標籤 0」表示應讀的字，角色不同。

從圖片讀出字元稱**光學字元辨識**（Optical Character Recognition，OCR）。可以用分類頭選 0 或 1，也可以讓文字核心逐步生成答案。分類選對類別就算對；生成若要求完整字串，`"00"` 比標籤多一字仍錯。這段只建立資料，沒有更新辨識器。

字形與標籤要人工核對，梯度不會發現畫圖器把 1 畫成 0。本例只改位移或粗細而未改字元時，答案標籤仍相同。每個字目前只有一個原形；若把數字 0 的原形與所有變體整組留出，訓練側就沒有 0 這個類別。要檢查熟悉類別的新原圖，需為每類準備多張獨立原圖，再把同一原圖及其變體當一個來源家族，整組放在訓練側或留出側。若用 RGB 入口，要把單通道小圖放到固定畫布、再明確複製通道，不能把 `(5,3)` 直接當 `(3,16,16)`。

既有 0–99 固定字形實驗按完整字串切分：同一字串的位移版本放在同一側。訓練側已包含 0–9 全部字元，留出側測的是熟悉字元的新字串組合。訓練代價降低，最後新字串完整讀對只有 2/30，留下「熟悉練習改善不等於新組合讀對」這個例子。這個實驗沒有測新字元類別、另一套字體或手寫。下一節再看整串與逐字判尺。

練習把 1 第一行 `010` 改 `110`，畫出多的左上筆畫，標籤仍為 1。改變字體外觀和改變需要辨識的字，是不同的資料操作。

<details>
<summary>回顧與查證</summary>

可回顧：[10.1像素tensor](https://birdhackor.github.io/tiny-perceptron-vlm/10.1.html)、[10.5分類監督](https://birdhackor.github.io/tiny-perceptron-vlm/10.5.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/ocr.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
